# MTPL frequency: model training

Fit and publish the raw model first. The optional later section loads level collapses exported from the scratch notebook's interactive editor.

In [ ]:
DATABASE_MODE = "local"  # "local" or "remote"
RUNTIME_MODULE = None
EXPECTED_REMOTE_DATABASE = ""
ALLOW_REMOTE_WRITES = False

In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)

from superglm import Categorical, Numeric, Spline, SuperGLM  # noqa: E402

from pricing_pipeline.models.config import ValidationSplitConfig  # noqa: E402
from pricing_pipeline.notebook import (  # noqa: E402
    PricingModelSpec,
    apply_level_groupings,
    build_candidate,
    connect,
    inspect_level_groupings,
    load_level_groupings,
    load_model_frame,
    publish_candidate,
    register_model,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/mtpl_frequency"
FRAME_ARTIFACT_PATH = MODEL_DIR / ".local" / "model_frame.joblib"
GROUPING_ARTIFACT_PATH = MODEL_DIR / ".local" / "routine_groupings.joblib"

## Connect and load the exact ingested frame

In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)
frame = load_model_frame(FRAME_ARTIFACT_PATH)
display({"Rows": len(frame), "Columns": len(frame.columns)})

## Stable model identity and validation decision

In [ ]:
MODEL_FEATURES = (
    "VehAge", "DrivAge", "BonusMalus", "LogDensity", "Area",
    "VehPower", "VehBrand", "VehGas", "Region",
)
MODEL = PricingModelSpec(
    name="MTPL_FREQ",
    label="Motor frequency",
    target="ClaimNb",
    model_type="superglm_poisson",
    deployment_slot="MTPL_FREQ_UAT",
    features=MODEL_FEATURES,
    dataset_name="freMTPL2freq_model_frame",
    source_system="freMTPL_raw_sql",
    pk_columns=("IDpol",),
    offset_column="LogExposure",
    offset_source_column="Exposure",
    offset_label="log(Exposure)",
    sample_weight_column=None,
    export_weight_column="Exposure",
    data_as_of_column="data_as_of",
    validation=ValidationSplitConfig.kfold(
        n_splits=5,
        random_state=42,
        shuffle=True,
    ),
)
model = register_model(pricing, MODEL, source_root=MODEL_DIR)

## Raw model: no pre-applied groupings and no editor

In [ ]:
RAW_FEATURES = {
    "VehAge": Spline(),
    "DrivAge": Spline(),
    "BonusMalus": Spline(),
    "LogDensity": Numeric(),
    "Area": Categorical(),
    "VehPower": Categorical(),
    "VehBrand": Categorical(),
    "VehGas": Categorical(),
    "Region": Categorical(),
}
raw_superglm_model = SuperGLM(
    family="poisson",
    selection_penalty=0.0,
    discrete=True,
    n_bins=256,
    features=RAW_FEATURES,
)

In [ ]:
raw_candidate = build_candidate(
    pricing,
    model=model,
    frame=frame,
    superglm_model=raw_superglm_model,
    model_kind="RAW",
)
raw_candidate.metrics

In [ ]:
raw_published = publish_candidate(pricing, raw_candidate)
display({
    "Model": raw_published.model_name,
    "Kind": raw_published.model_kind,
    "Package": raw_published.package_version,
    "Manifest": raw_published.manifest_id,
    "State": raw_published.package_status,
    "Reused equivalent": raw_published.deduplicated,
})

## Optional routine edit: editor-exported simplifications

`99_scratch_work.ipynb` exports all interactive categorical collapses from a reviewed RAW candidate. This section loads the actual `LevelGrouping` objects and automatically skips `ROUTINE_EDIT` when the artifact is absent or empty.

In [ ]:
LEVEL_GROUPINGS = (
    load_level_groupings(
        GROUPING_ARTIFACT_PATH,
        frame=frame,
        model=model,
    )
    if GROUPING_ARTIFACT_PATH.is_file()
    else {}
)
ROUTINE_EDIT_CONFIGURED = bool(LEVEL_GROUPINGS)
routine_superglm_model = None
if ROUTINE_EDIT_CONFIGURED:
    ROUTINE_FEATURES = apply_level_groupings(RAW_FEATURES, LEVEL_GROUPINGS)
    routine_superglm_model = SuperGLM(
        family="poisson",
        selection_penalty=0.0,
        discrete=True,
        n_bins=256,
        features=ROUTINE_FEATURES,
    )
    display(inspect_level_groupings(GROUPING_ARTIFACT_PATH))
else:
    display("No exported level collapses: ROUTINE_EDIT skipped.")

In [ ]:
routine_published = None
if ROUTINE_EDIT_CONFIGURED:
    routine_candidate = build_candidate(
        pricing,
        model=model,
        frame=frame,
        superglm_model=routine_superglm_model,
        model_kind="ROUTINE_EDIT",
    )
    display(routine_candidate.metrics)
    routine_published = publish_candidate(pricing, routine_candidate)
    display({
        "Kind": routine_published.model_kind,
        "Package": routine_published.package_version,
        "Manifest": routine_published.manifest_id,
        "State": routine_published.package_status,
        "Reused equivalent": routine_published.deduplicated,
    })